In [10]:
import anndata as ad
import pandas as pd
import scanpy as sc
import numpy as np
import json

from skimage import io
from starfysh import utils

adata_raw = sc.read_h5ad("../v2-analysis/slv14.h5ad")
adata_raw

print(adata_raw)


AnnData object with n_obs × n_vars = 28038 × 18085
    obs: 'orig.ident', 'nCount_Spatial', 'nFeature_Spatial', 'percent.mt'
    obsm: 'spatial'


In [11]:
adata = adata_raw.copy()

# Remove mitochondrial and ribosomal genes
mt = adata.var_names.str.startswith("MT-")
rb = adata.var_names.str.startswith("RP")

adata = adata[:, ~(mt | rb)].copy()

# Normalize and log-transform
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

# Identify highly variable genes
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat",
    n_top_genes=2000
)

In [12]:
adata_raw = adata_raw[
    adata.obs_names,
    adata.var_names
].copy()

adata_raw.var["highly_variable"] = adata.var["highly_variable"]

In [13]:
gene_sig = pd.read_csv(
    "../v2-analysis/GVHD_spatial_signature.csv"
)

gene_sig.head()

for column in gene_sig.columns:
    gene_sig[column] = gene_sig[column].where(
        gene_sig[column].isin(adata.var_names)
    )

gene_sig

,Plasma cells,Myeloid,Fibroblast,B cells,NK cells,Instestinal Epithelial cells,Enterocyte,Stomach Epithelial cells,Stomach Stem cells,Endothelial cells,...,CD4+ Effector T cells,CD4+ Central Memory T cells,CD8+ Cytotoxic Unconventional T cells,CD8+ Proliferating T cells,CD4+ Regulatory T cells,CD8+ Homeostatic Unconventional T cells,CD8+ Tissue Resident Memory T cells,CD8+ Transitioning Resident T cells,Intestine Epithelial Stem cells,Stomach Body Epithelial cells
0,DERL3,CD14,ACTA2,CD19,NCR1,EPCAM,ANPEP,CLDN18,LGR5,PECAM1,...,CD3E,CD3E,CD3E,CD3E,CD3E,CD3E,CD3E,CD3E,LGR5,LIPF
1,DNAJB9,CD163,C1S,CR2,NCAM1,TFF3,PRAP1,PGC,MUC6,FLT1,...,CD3G,CD3G,CD3G,CD3G,CD3G,CD3G,CD3G,CD3G,ASCL2,PGA5
2,FCRL5,AIF1,COL1A1,IGHD,KRT86,C10orf99,RBP2,PSCA,FMOD,HSPG2,...,CD3D,CD3D,CD3D,CD3D,CD3D,CD3D,CD3D,CD3D,SMOC2,CHIA
3,NaN,C1QA,COL1A2,MS4A1,SH2D1B,CKB,FABP2,CTSE,FUT9,RAMP3,...,CD40LG,CD4,GZMA,MKI67,CD4,KIR2DL4,CD8A,CD8A,MSI1,ATP4A
4,MZB1,C1QB,COL3A1,PAX5,TXK,CEACAM5,SMIM24,ANXA10,GP2,RAMP2,...,CD4,TNFRSF4,GNLY,GTSE1,FOXP3,HOPX,NaN,NaN,RARRES2,ATP4B
5,IGLC1,C1QC,COL6A1,BANK1,NaN,AXIN2,CCL25,TFF2,MSMB,PLVAP,...,IL7R,CD40LG,CD7,TOP2A,TNFRSF4,TMIGD2,TMIGD2,TMIGD2,SOX9,ALDOB
6,TNFRSF17,CSF1R,COL6A2,VPREB3,NaN,PLA2G2A,CLDN15,VSIG1,MUC5AC,SLCO2A1,...,KLRB1,CCR4,FCER1G,CD8A,CTLA4,CD7,CD248,CD248,CENPM,CCKBR
7,TXNDC5,IGSF6,CXCL14,NaN,NaN,PRAC1,NaN,TFF1,TFF1,SOX18,...,SESN1,CD28,NKG7,NaN,TBC1D4,NaN,ITGA1,ITGA1,EPHB2,CPA2
8,SDC1,MERTK,CYGB,NaN,NaN,MUC4,REEP6,MUC5AC,PGC,VWF,...,CCR6,IL7R,CD247,LAG3,TIGIT,CD160,NaN,NaN,OLFM4,DNER
9,EAF2,MS4A6A,DCN,NaN,CD160,SNORC,KHK,CA9,TFF2,NaN,...,TNF,TCF7,KIR2DL4,HAVCR2,IL2RA,ITGA1,CD101,CD101,NaN,DRD5


In [17]:
spatial_dir = "../data/SLV14/binned_outputs/square_016um/spatial"

positions = pd.read_parquet(
    f"{spatial_dir}/tissue_positions.parquet"
)

with open(f"{spatial_dir}/scalefactors_json.json") as f:
    scalefactor = json.load(f)

img = io.imread(
    f"{spatial_dir}/tissue_hires_image.png"
)

positions = positions.set_index("barcode")
positions = positions.loc[adata.obs_names].copy()

positions.head()

map_info = positions[
    [
        "array_row",
        "array_col",
        "pxl_col_in_fullres",
        "pxl_row_in_fullres"
    ]
].copy()

map_info.columns = [
    "array_row",
    "array_col",
    "imagecol",
    "imagerow"
]

img_metadata = {
    "map_info": map_info,
    "scalefactor": scalefactor,
    "img": img
}

print(map_info.shape)
print(adata.shape)
print(img.shape)
print(scalefactor)

args = utils.VisiumArguments(
    adata_raw,
    adata,
    gene_sig,
    img_metadata,
    sample_id="SLV14"
)


(28038, 4)
(28038, 18032)
(4261, 6000, 3)
{'spot_diameter_fullres': 28.83691815401692, 'bin_size_um': 16.0, 'microns_per_pixel': 0.5548443115365029, 'tissue_lowres_scalef': 0.05097273, 'fiducial_diameter_fullres': 594.7614369265991, 'tissue_hires_scalef': 0.5097273, 'regist_target_img_scalef': 0.5097273}


/Users/evangrosso/Library/r-miniconda-arm64/envs/starfysh/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
[2026-08-21 13:04:43] Subsetting highly variable & signature genes ...
/Users/evangrosso/Library/r-miniconda-arm64/envs/starfysh/lib/python3.10/site-packages/starfysh/utils.py:251: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  self.adata.uns['spatial'] = {
/Users/evangrosso/Library/r-miniconda-arm64/envs/starfysh/lib/python3.10/site-packages/starfysh/utils.py:258: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  self.adata_norm.uns['spatial'] = {
[2026-08-21 13:04:43] Smoothing library size by taking averaging with neighbor spots...
[2026-08-21 13:04:51] Retrieving & normalizing signature gene expressions...
[2026-08-21 13:05:37] Identifying anchor spots (highly expression 

In [ ]:
anchors = args.get_anchors()
anchors.head()

anchors.notna().sum()

import inspect


print(inspect.signature(utils.run_starfysh))

/Users/evangrosso/Library/r-miniconda-arm64/envs/starfysh/lib/python3.10/site-packages/starfysh/utils.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return anchors_df.applymap(


(visium_args, n_repeats=3, lr=0.0001, epochs=100, batch_size=32, alpha_mul=50, poe=False, device=device(type='cpu'), seed=0, verbose=True)


In [21]:
import torch

# Use CPU for compatibility
device = torch.device("cpu")

# Fix Starfysh / SciPy sparse-matrix compatibility
if not isinstance(args.adata.X, np.ndarray):
    args.adata.X = args.adata.X.toarray()

if not isinstance(args.adata_norm.X, np.ndarray):
    args.adata_norm.X = args.adata_norm.X.toarray()

# Run Starfysh
model, losses, adata_out = utils.run_starfysh(
    args,
    n_repeats=3,
    lr=1e-4,
    epochs=100,
    batch_size=32,
    alpha_mul=50,
    poe=False,
    device=device,
    seed=0,
    verbose=True
)

[2026-08-21 13:24:54] Running Starfysh with 3 restarts, choose the model with best parameters...
[2026-08-21 13:24:54] Initializing model parameters...
/Users/evangrosso/Library/r-miniconda-arm64/envs/starfysh/lib/python3.10/site-packages/starfysh/dataloader.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  torch.Tensor(self.gexp.iloc[idx, :]),  # normalized signature exprs
[2026-08-21 13:27:58] Epoch[10/100], train_loss: 195.3516, train_reconst: 68.7936, train_u: 15.3060,train_z: 5.0255,train_c: 105.1420,train_l: 1.0847
[2026-08-21 13:31:04] Epoch[20/100], train_loss: 176.1812, train_reconst: 67.3100, train_u: 13.5073,train_z: 5.2930,train_c: 88.9863,train_l: 1.0846
[2026-08-21 13:34:21] Epoch[30/100], train_loss: 164.8289, train_reconst: 66.6386, train_u: 12.2308,train_z: 5.4324,train_c: 79.4384,

ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
print(adata_out)
print(adata_out.obs.columns.tolist())
print(adata_out.obsm.keys())
print(adata_out.uns.keys())